In [1]:
import pandas as pd 

In [2]:
cpi = pd.read_csv('Data\common_player_info.csv')
game = pd.read_csv('Data\game.csv')
ls = pd.read_csv('Data\line_score.csv')
os = pd.read_csv('Data\other_stats.csv')

# Player Info

In [3]:
# rename person_id to player_id in cpi for better readability 
cpi2 = cpi.rename(columns={'person_id': 'player_id'})

In [4]:
# drop unnecessary columns in cpi2
player_df = cpi2.drop(columns=['display_last_comma_first', 'display_fi_last'])

# ensure is unique players 
player_df['player_id'].is_unique

True

# Team Info

In [5]:
game.columns

Index(['season_id', 'team_id_home', 'team_abbreviation_home', 'team_name_home',
       'game_id', 'game_date', 'matchup_home', 'wl_home', 'min', 'fgm_home',
       'fga_home', 'fg_pct_home', 'fg3m_home', 'fg3a_home', 'fg3_pct_home',
       'ftm_home', 'fta_home', 'ft_pct_home', 'oreb_home', 'dreb_home',
       'reb_home', 'ast_home', 'stl_home', 'blk_home', 'tov_home', 'pf_home',
       'pts_home', 'plus_minus_home', 'video_available_home', 'team_id_away',
       'team_abbreviation_away', 'team_name_away', 'matchup_away', 'wl_away',
       'fgm_away', 'fga_away', 'fg_pct_away', 'fg3m_away', 'fg3a_away',
       'fg3_pct_away', 'ftm_away', 'fta_away', 'ft_pct_away', 'oreb_away',
       'dreb_away', 'reb_away', 'ast_away', 'stl_away', 'blk_away', 'tov_away',
       'pf_away', 'pts_away', 'plus_minus_away', 'video_available_away',
       'season_type'],
      dtype='object')

In [6]:
ls.columns

Index(['game_date_est', 'game_sequence', 'game_id', 'team_id_home',
       'team_abbreviation_home', 'team_city_name_home', 'team_nickname_home',
       'team_wins_losses_home', 'pts_qtr1_home', 'pts_qtr2_home',
       'pts_qtr3_home', 'pts_qtr4_home', 'pts_ot1_home', 'pts_ot2_home',
       'pts_ot3_home', 'pts_ot4_home', 'pts_ot5_home', 'pts_ot6_home',
       'pts_ot7_home', 'pts_ot8_home', 'pts_ot9_home', 'pts_ot10_home',
       'pts_home', 'team_id_away', 'team_abbreviation_away',
       'team_city_name_away', 'team_nickname_away', 'team_wins_losses_away',
       'pts_qtr1_away', 'pts_qtr2_away', 'pts_qtr3_away', 'pts_qtr4_away',
       'pts_ot1_away', 'pts_ot2_away', 'pts_ot3_away', 'pts_ot4_away',
       'pts_ot5_away', 'pts_ot6_away', 'pts_ot7_away', 'pts_ot8_away',
       'pts_ot9_away', 'pts_ot10_away', 'pts_away'],
      dtype='object')

In [7]:
# renaming 'All star' season type values to be standardized 
game2 = game.copy() 
game2['season_type'] = game2['season_type'].replace('All Star', 'All-Star')
game2['season_type'].value_counts()

Regular Season    60192
Playoffs           3842
Pre Season         1536
All-Star            128
Name: season_type, dtype: int64

In [8]:
# dropping duplicate game ids in game df
game2 = game2.drop_duplicates(subset='game_id', keep='first')
game2['game_id'].is_unique

True

In [9]:
# dropping duplicate game ids in line score df
ls2 = ls.drop_duplicates(subset='game_id', keep='first')
ls2['game_id'].is_unique

True

In [10]:
# printing duplicated columns in game3 and ls2 to drop before merging 
print('Similar columns:')
for col in game2.columns:
    if col in ls2.columns:
        print(col)

# dropping all similar columns EXCEPT game_id fro ls2 (bc is smaller df) 
ls2 = ls2.drop(columns=['team_id_home', 'team_abbreviation_home', 'pts_home', 'team_id_away', 'team_abbreviation_away', 'pts_away'])

# verify dropping 
print('\nChecking columns:')
for col in game2.columns:
    if col in ls2.columns:
        print(col)

Similar columns:
team_id_home
team_abbreviation_home
game_id
pts_home
team_id_away
team_abbreviation_away
pts_away

Checking columns:
game_id


In [11]:
# merging game2 and ls2 on game_id to have one game df
game_df = pd.merge(game2, ls2, on='game_id', how='inner')
print(f'game2 shape: {game2.shape}')
print(f'ls2 shape: {ls2.shape}')
print(f'game_df shape: {game_df.shape}')
game_df 

game2 shape: (65642, 55)
ls2 shape: (58013, 37)
game_df shape: (58013, 91)


,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,...,pts_ot1_away,pts_ot2_away,pts_ot3_away,pts_ot4_away,pts_ot5_away,pts_ot6_away,pts_ot7_away,pts_ot8_away,pts_ot9_away,pts_ot10_away
0,21946,1610610035,HUS,Toronto Huskies,24600001,1946-11-01 00:00:00,HUS vs. NYK,L,0,25.0,...,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,21946,1610610034,BOM,St. Louis Bombers,24600003,1946-11-02 00:00:00,BOM vs. PIT,W,0,20.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,21946,1610610032,PRO,Providence Steamrollers,24600002,1946-11-02 00:00:00,PRO vs. BOS,W,0,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,21946,1610610025,CHS,Chicago Stags,24600004,1946-11-02 00:00:00,CHS vs. NYK,W,0,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,21946,1610610028,DEF,Detroit Falcons,24600005,1946-11-02 00:00:00,DEF vs. WAS,L,0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58008,42022,1610612743,DEN,Denver Nuggets,42200402,2023-06-04 00:00:00,DEN vs. MIA,L,240,39.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
58009,42022,1610612748,MIA,Miami Heat,42200403,2023-06-07 00:00:00,MIA vs. DEN,L,240,34.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
58010,42022,1610612748,MIA,Miami Heat,42200404,2023-06-09 00:00:00,MIA vs. DEN,L,240,35.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
58011,42022,1610612743,DEN,Denver Nuggets,42200405,2023-06-12 00:00:00,DEN vs. MIA,W,240,38.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
# dropping irrelevant columns in other stats 
os2 = os.drop(columns=['league_id'])

# dropping duplicate game ids in other stats df
os2 = os2.drop_duplicates(subset='game_id', keep='first')
os2['game_id'].is_unique

True

In [17]:
# printing duplicated columns in game_df and os2 to drop before merging 
print('Similar columns:')
for col in game_df.columns:
    if col in os2.columns:
        print(col)

# dropping all similar columns EXCEPT game_id fro ls2 (bc is smaller df) 
os2 = os2.drop(columns=['team_id_home', 'team_abbreviation_home', 'team_id_away', 'team_abbreviation_away'])

# # verify dropping 
print('\nChecking columns:')
for col in game_df.columns:
    if col in os2.columns:
        print(col)

Similar columns:
team_id_home
team_abbreviation_home
game_id
team_id_away
team_abbreviation_away

Checking columns:
game_id


In [18]:
# merging game_df and os2 on game_id to have one game df with all stats
final_game_df = pd.merge(game_df, os2, on='game_id', how='inner')
print(f'game_df shape: {game_df.shape}')
print(f'os2 shape: {os2.shape}')
print(f'final_game_df shape: {final_game_df.shape}')
final_game_df

game_df shape: (58013, 91)
os2 shape: (28261, 21)
final_game_df shape: (28261, 111)


,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,...,pts_off_to_home,team_city_away,pts_paint_away,pts_2nd_chance_away,pts_fb_away,largest_lead_away,team_turnovers_away,total_turnovers_away,team_rebounds_away,pts_off_to_away
0,21996,1610612747,LAL,Los Angeles Lakers,29600012,1996-11-01 00:00:00,LAL vs. PHX,W,240,31.0,...,NaN,Los Angeles,42,10,13,19,0.0,23.0,11.0,NaN
1,21996,1610612748,MIA,Miami Heat,29600005,1996-11-01 00:00:00,MIA vs. ATL,W,240,35.0,...,NaN,Miami,32,15,14,16,1.0,19.0,6.0,NaN
2,21996,1610612751,NJN,New Jersey Nets,29600002,1996-11-01 00:00:00,NJN vs. CLE,L,240,23.0,...,NaN,New Jersey,26,16,4,2,1.0,22.0,12.0,NaN
3,21996,1610612765,DET,Detroit Pistons,29600007,1996-11-01 00:00:00,DET vs. IND,W,240,32.0,...,NaN,Detroit,30,14,7,9,2.0,19.0,10.0,NaN
4,21996,1610612744,GSW,Golden State Warriors,29600013,1996-11-01 00:00:00,GSW vs. LAC,L,240,27.0,...,NaN,Golden State,30,9,2,6,0.0,20.0,7.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28256,42022,1610612743,DEN,Denver Nuggets,42200401,2023-06-01 00:00:00,DEN vs. MIA,W,240,40.0,...,13.0,Denver,46,7,9,24,0.0,10.0,7.0,9.0
28257,42022,1610612743,DEN,Denver Nuggets,42200402,2023-06-04 00:00:00,DEN vs. MIA,L,240,39.0,...,19.0,Miami,34,11,5,12,0.0,11.0,4.0,23.0
28258,42022,1610612748,MIA,Miami Heat,42200403,2023-06-07 00:00:00,MIA vs. DEN,L,240,34.0,...,8.0,Denver,60,14,11,21,1.0,14.0,7.0,17.0
28259,42022,1610612748,MIA,Miami Heat,42200404,2023-06-09 00:00:00,MIA vs. DEN,L,240,35.0,...,8.0,Miami,46,9,14,3,1.0,15.0,9.0,17.0


# Saving csvs

In [19]:
# save player_df as csv file to upload 
player_df.to_csv('Data/player_df.csv', index=False)

# save final_game_df as csv file to upload
final_game_df.to_csv('Data/game_df.csv', index=False)